# Part 2 - Osteotomy Site Detection Pipeline

## 4.3 Task 1: Inspect the dataset and summarise it

In [1]:
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("../dataset")
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"

image_files = sorted(IMAGES_DIR.glob("*.jpg"))
label_files = sorted(LABELS_DIR.glob("*.txt"))

image_stems = {f.stem for f in image_files}
label_stems = {f.stem for f in label_files}

print(f"Images: {len(image_files)}")
print(f"Labels: {len(label_files)}")
print(f"Images without a matching label: {sorted(image_stems - label_stems)}")
print(f"Labels without a matching image: {sorted(label_stems - image_stems)}")

Images: 820
Labels: 820
Images without a matching label: []
Labels without a matching image: []


Parse the `patientID_sessionNumber_sliceNumber` filename convention to count unique patients and imaging sessions.

In [2]:
patients = set()
sessions = set()
images_per_patient = Counter()
sessions_per_patient = {}

for f in image_files:
    patient_id, session_num, slice_num = f.stem.split("_")
    patients.add(patient_id)
    sessions.add((patient_id, session_num))
    images_per_patient[patient_id] += 1
    sessions_per_patient.setdefault(patient_id, set()).add(session_num)

print(f"Unique patients: {len(patients)}")
print(f"Unique (patient, session) pairs: {len(sessions)}")

n_sessions_dist = Counter(len(v) for v in sessions_per_patient.values())
print(f"Patients by number of sessions: {dict(sorted(n_sessions_dist.items()))}")

counts = list(images_per_patient.values())
print(f"Images per patient: min={min(counts)}, max={max(counts)}, "
      f"mean={sum(counts)/len(counts):.1f}, median={sorted(counts)[len(counts)//2]}")
print(f"Top 5 patients by image count: {images_per_patient.most_common(5)}")

Unique patients: 85
Unique (patient, session) pairs: 160
Patients by number of sessions: {1: 31, 2: 34, 3: 19, 4: 1}
Images per patient: min=1, max=123, mean=9.6, median=5
Top 5 patients by image count: [('8', 123), ('3', 112), ('18', 65), ('500', 30), ('403', 28)]


Inspect the annotations themselves: box count per image (are there empty/negative slices, multi-box slices?) and which classes are used.

In [3]:
boxes_per_image = {}
class_ids = Counter()

for f in label_files:
    lines = [l for l in f.read_text().splitlines() if l.strip()]
    boxes_per_image[f.stem] = len(lines)
    for line in lines:
        class_ids[int(line.split()[0])] += 1

box_count_dist = Counter(boxes_per_image.values())
print(f"Distribution of boxes per image: {dict(sorted(box_count_dist.items()))}")
print(f"Class IDs used and box counts: {dict(class_ids)}")
print(f"Total annotated boxes: {sum(class_ids.values())}")
print(f"Images with zero boxes (negative slices): {box_count_dist.get(0, 0)}")

Distribution of boxes per image: {0: 1, 1: 605, 2: 140, 3: 72, 4: 2}
Class IDs used and box counts: {0: 1109}
Total annotated boxes: 1109
Images with zero boxes (negative slices): 1


## 4.3 Task 2: Train / validation / test splits

The split is built at the **patient level**: every image belonging to a given
patient (across all of that patient's sessions) is assigned entirely to one
split. This prevents leakage - without it, near-identical slices of the same
join (adjacent slice numbers, or the same join re-scanned at a later session)
could appear in both training and test, letting the model "recognise" a specific
patient's anatomy or hardware rather than genuinely learning to localise
osteotomy sites, and inflating the test score.

Because images per patient are heavily skewed (1 to 123, see Task 1), a plain
random assignment of patients to splits is risky: a single unlucky draw could put
a 100+ image patient entirely into test and badly distort the intended split
sizes. Instead, patients are assigned with a greedy largest-first heuristic
targeting a 70 / 15 / 15 split by **image count**: patients are shuffled (fixed
seed, for reproducibility) then sorted by image count descending, and each
patient in turn is assigned to whichever split is currently furthest below its
target share of total images. This keeps the realised split close to 70/15/15
while still assigning whole patients.

In [4]:
import random

random.seed(42)
TARGET_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

patient_items = list(images_per_patient.items())
random.shuffle(patient_items)
patient_items.sort(key=lambda x: x[1], reverse=True)

total_images = sum(images_per_patient.values())
split_image_counts = {k: 0 for k in TARGET_RATIOS}
split_patients = {k: [] for k in TARGET_RATIOS}

for patient_id, count in patient_items:
    deficits = {
        split: TARGET_RATIOS[split] * total_images - split_image_counts[split]
        for split in TARGET_RATIOS
    }
    chosen = max(deficits, key=deficits.get)
    split_patients[chosen].append(patient_id)
    split_image_counts[chosen] += count

for split in TARGET_RATIOS:
    pct = split_image_counts[split] / total_images * 100
    print(f"{split}: {len(split_patients[split])} patients, "
          f"{split_image_counts[split]} images ({pct:.1f}%)")

assert set().union(*split_patients.values()) == patients
assert sum(len(v) for v in split_patients.values()) == len(patients)

train: 35 patients, 574 images (70.0%)
val: 25 patients, 123 images (15.0%)
test: 25 patients, 123 images (15.0%)


**Sanity check.** Because the greedy assignment processes patients largest-first and
always fills whichever split has the biggest current deficit, and train's target
(70%) starts far larger than val's or test's, train absorbs essentially all of the
heavy patients first. The result: val and test end up composed of many small,
diverse patients (each contributing at most 11 images) rather than being dominated
by one or two large patients - confirmed below, and important because an eval split
that is secretly "mostly one patient" would not be a meaningful measure of
generalisation.

In [5]:
for split in TARGET_RATIOS:
    patient_counts = sorted(
        [(p, images_per_patient[p]) for p in split_patients[split]],
        key=lambda x: -x[1],
    )
    print(f"{split}: largest contributing patient = {patient_counts[0]}, "
          f"smallest = {patient_counts[-1]}")

# persist the split so the training notebook uses the exact same assignment
import json as _json
split_map = {p: s for s, plist in split_patients.items() for p in plist}
SPLIT_FILE = Path("../dataset/splits.json")
_json.dump(split_map, open(SPLIT_FILE, "w"), indent=1)
print(f"\nSaved patient->split mapping for {len(split_map)} patients to {SPLIT_FILE.resolve()}")

train: largest contributing patient = ('8', 123), smallest = ('116', 1)
val: largest contributing patient = ('1052', 11), smallest = ('1281', 1)
test: largest contributing patient = ('920', 10), smallest = ('104', 1)

Saved patient->split mapping for 85 patients to E:\Bone Union Detection\dataset\splits.json
